<a href="https://colab.research.google.com/github/imagra93/ML-course-labs/blob/main/machine_learning/lab%20-%20Ordinary%20Least%20Squares%20Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ordinary Least Squares (OLS) Linear Regression from Scratch

**Theory:** [notes_ML/01_linear_regression.ipynb](../notes_ML/01_linear_regression.ipynb) (loss, gradient, gradient descent, learning rate, normal equation, MLE, metrics). This lab only implements it.

We fit the model $h_{\boldsymbol{\theta}}(\mathbf{x}) = \boldsymbol{\theta}^\top\mathbf{x} = \theta_0 + \theta_1 x_1$ to toy data, step by step.

### Learning Objectives
- Implement the MSE cost function from scratch
- Implement batch gradient descent
- Implement the closed-form Normal Equation and compare
- Vectorize the implementation for efficiency

<div align='center'>

![Linear Regression](https://upload.wikimedia.org/wikipedia/commons/3/3a/Linear_regression.svg)

*Linear regression finds the line (or hyperplane) that best fits the data by minimizing the sum of squared residuals.*

</div>

In [ ]:
%matplotlib inline
from IPython.display import HTML
import numpy as np
import matplotlib.pyplot as plt
import time

plt.style.use('seaborn-v0_8-whitegrid')

#### Generate the data
A toy dataset: hours studied ($x$) vs. test score ($y$), generated as $y = 1 + 2x + \text{noise}$.

In [ ]:
# Generate random data
np.random.seed(0)
m = 1000
X = np.random.rand(m, 1) * 10                   # 1000 data points between 0 and 10
y = 2 * X + 1 + np.random.randn(m, 1) * 2       # target values with some noise

m, n = X.shape

print(f"Num features : {n}")
print(f"Num samples  : {m}")

In [ ]:
# y has shape (m, 1); we want a flat vector of shape (m,)
print(f"y shape before squeeze: {y.shape}")
y = y.squeeze()
print(f"y shape after squeeze : {y.shape}")

Let's plot the data.

In [ ]:
# Plot the dataset
plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.6, edgecolors='white', linewidths=0.5,
            color='steelblue', label='Training data')
plt.xlabel('Hours of study')
plt.ylabel('Test scores')
plt.title('Dataset: Hours Studied vs. Test Scores')
plt.legend()
plt.tight_layout()
plt.show()

Before continuing, we add a first column of 1's ($x_0 = 1$, for the intercept $\theta_0$). This gives the design matrix $\mathbf{X}$ of shape $m \times (n+1)$.

In [ ]:
print(f"X shape before augmentation : {X.shape}")
X = np.concatenate([np.ones((m, 1)), X], axis=1)
print(f"X shape after augmentation  : {X.shape}")
print("First 5 rows of X:")
print(X[:5, :])

#### Defining the hyperparamters

In [ ]:
# Hyperparameters
learning_rate = 0.01
initial_theta_vector = np.zeros(X.shape[1])
num_iterations = 100

print("-" * 40)
print("Hyperparameter Configuration")
print("-" * 40)
print(f"  Learning rate    : {learning_rate}")
print(f"  Num iterations   : {num_iterations}")
print(f"  Initial theta    : {initial_theta_vector}")
print("-" * 40)

Could you have used a different initial_theta_vector?


#### Define the Cost Function

Mean Squared Error (MSE):

$$\mathcal{J}(\boldsymbol{\theta}) = \frac{1}{2m}\sum_{i=1}^m\bigl(h_{\boldsymbol{\theta}}(\mathbf{x}^{(i)}) - y^{(i)}\bigr)^2$$

For every example $i$: **1)** predict $\hat y^{(i)} = \boldsymbol{\theta}^\top\mathbf{x}^{(i)}$, **2)** compute the error $\hat y^{(i)} - y^{(i)}$, **3)** square it and add it to the total. At the end divide by $2m$.

In [ ]:
def compute_cost(theta_vector, X, y):
    """Compute MSE cost (loop-based) for parameter vector theta_vector."""
    total_cost = 0

    # number of datapoints in training data
    m = len(y)

    # Compute sum of squared errors (without the 1/2m term)
    for i in range(m):
        x_i = X[i]                           # i-th example: [1, x_1]
        prediction = theta_vector @ x_i      # 1) h(x) = theta^T x = theta_0 * 1 + theta_1 * x_1
        error = prediction - y[i]            # 2) how far we are from the true value
        total_cost += error ** 2             # 3) squared error

    # Return average of squared error
    return total_cost / (2 * m)

Check your compute_cost implementetation.

In [ ]:
initial_cost = compute_cost(initial_theta_vector, X, y)
print(f"Initial cost (should be ~78.6) : {initial_cost:.4f}")


#### Gradient Descent

Partial derivative of the cost with respect to each parameter $\theta_j$:

$$\frac{\partial \mathcal{J}}{\partial \theta_j} = \frac{1}{m}\sum_{i=1}^m \bigl(h_{\boldsymbol{\theta}}(\mathbf{x}^{(i)}) - y^{(i)}\bigr)\, x_j^{(i)}$$

**Update rule** (all $\theta_j$ at the same time): $\;\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \alpha \nabla_{\boldsymbol{\theta}}\mathcal{J}(\boldsymbol{\theta})$

For every example $i$: **1)** compute the error $\hat y^{(i)} - y^{(i)}$, **2)** add $\text{error} \cdot x_j^{(i)}$ to the gradient of each $\theta_j$. Then divide by $m$ and take the step.

<div align='center'>

![Gradient Descent](https://upload.wikimedia.org/wikipedia/commons/f/ff/Gradient_descent.svg)

*Gradient descent iteratively moves the parameters in the direction of steepest descent of the cost surface until convergence.*

</div>

In [ ]:
def step_gradient(theta_current, X, y, alpha):
    """takes one step down towards the minima

    Args:
        theta_current (float vector): current values of theta
        X (np.array): training features (with the column of ones)
        y (np.array): training targets
        alpha (float): learning rate / step size

    Returns:
        theta_updated (float vector): new values of theta
    """

    theta_gradient = np.zeros(X.shape[1])
    m = len(y)

    # Calculate Gradient
    for i in range(m):
        x_i = X[i]                                   # i-th example: [1, x_1]
        error = theta_current @ x_i - y[i]           # 1) prediction - true value
        for j in range(len(theta_current)):          # 2) one partial derivative per theta_j
            theta_gradient[j] += error * x_i[j]

    theta_gradient /= m

    # Update current theta vector
    theta_updated = theta_current - alpha * theta_gradient

    #Return updated parameters
    return theta_updated

In [ ]:
theta_updated = step_gradient(initial_theta_vector, X, y, learning_rate)
print(f"Theta after one step (should be [0.1098, 0.7116]) : {theta_updated}")

And run steps of the gradient descent for as many iterations as wanted.

In [ ]:
def gradient_descent(X, y, initial_theta_vector, learning_rate, num_iterations):
    """runs gradient descent

    Args:
        X (np.array): training data (features)
        y (np.array): training data pairs (outputs)
        initial_theta_vector (np.array): initial values of theta (random)
        learning_rate (float): hyperparameter to adjust the step size during descent
        num_iterations (int): hyperparameter, decides the number of iterations for which gradient descent would run

    Returns:
        tuple : (optimized_theta, cost_graph, theta_history)
            - optimized_theta : parameter vector at convergence
            - cost_graph      : list of MSE cost at each iteration
            - theta_history   : (num_iterations+1, 2) array of theta values,
                                starting from initial_theta_vector
    """

    # initial values
    theta_vector = initial_theta_vector.copy()

    # to store the cost and theta after each iteration
    cost_graph    = []
    theta_history = [theta_vector.copy()]

    # For every iteration, optimize the theta vector and compute its cost
    for i in range(num_iterations):
        cost_graph.append(compute_cost(theta_vector, X, y))
        theta_vector = step_gradient(theta_vector, X, y, learning_rate)
        theta_history.append(theta_vector.copy())

    return (theta_vector, cost_graph, np.array(theta_history))

#### Run gradient_descent() to get optimized parameters of theta

In [ ]:
optimized_theta_vector, cost_graph, theta_history = gradient_descent(
    X, y, initial_theta_vector, learning_rate, num_iterations
)

final_cost = compute_cost(optimized_theta_vector, X, y)

print("-" * 45)
print("Gradient Descent Results")
print("-" * 45)
print(f"  theta_0 (intercept) : {optimized_theta_vector[0]:.6f}")
print(f"  theta_1 (slope)     : {optimized_theta_vector[1]:.6f}")
print(f"  Minimised cost (MSE): {final_cost:.6f}")
print("-" * 45)

#### Plotting the cost per iterations

In [ ]:
final_cost_val = cost_graph[-1]

plt.figure(figsize=(8, 5))
plt.plot(cost_graph, color='steelblue', linewidth=2, label='Training cost')
plt.axhline(y=final_cost_val, color='crimson', linestyle='--', linewidth=1.2,
            label=f'Final cost = {final_cost_val:.4f}')
plt.annotate(
    f'{final_cost_val:.4f}',
    xy=(len(cost_graph) - 1, final_cost_val),
    xytext=(-60, 12),
    textcoords='offset points',
    color='crimson',
    fontsize=10,
    arrowprops=dict(arrowstyle='->', color='crimson', lw=1.2)
)
plt.xlabel('Iteration')
plt.ylabel('MSE Cost')
plt.title('Cost Convergence During Gradient Descent')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Cost-surface contour + Gradient Descent path ──────────────────────────
# Build a (theta_0, theta_1) grid and evaluate J on it
t0_range = np.linspace(-0.5, 2.0, 300)
t1_range = np.linspace(-0.5, 6.0, 300)
T0, T1 = np.meshgrid(t0_range, t1_range)
J_grid = np.array([
    compute_cost(np.array([t0, t1]), X, y)
    for t0, t1 in zip(T0.ravel(), T1.ravel())
]).reshape(T0.shape)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
cp = ax.contourf(T0, T1, J_grid, levels=50, cmap='viridis_r', alpha=0.88)
ax.contour(T0, T1, J_grid, levels=20, colors='white', linewidths=0.4, alpha=0.5)
plt.colorbar(cp, ax=ax, label=r'MSE Cost $\mathcal{J}(\boldsymbol{\theta})$')

# Gradient descent trajectory
ax.plot(theta_history[:, 0], theta_history[:, 1],
        color='crimson', linewidth=1.8, zorder=3, label='GD path')
ax.scatter(theta_history[:, 0], theta_history[:, 1],
           c='crimson', s=18, zorder=4)

# Mark start and end
ax.plot(*theta_history[0],  'o', color='yellow', markersize=12,
        zorder=5, label=f'Start  $\\theta$={theta_history[0].round(2)}')
ax.plot(*theta_history[-1], '*', color='white',  markersize=16,
        zorder=5, label=f'Optimum $\\theta$={theta_history[-1].round(2)}')

ax.set_xlabel(r'$\theta_0$ (intercept)', fontsize=12)
ax.set_ylabel(r'$\theta_1$ (slope)',     fontsize=12)
ax.set_title(r'Cost Surface $\mathcal{J}(\theta_0,\,\theta_1)$ with Gradient Descent Path',
             fontsize=13)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

The contour plot above shows the MSE cost surface as a function of $(\theta_0, \theta_1)$. Darker regions correspond to lower cost. The **red path** traces the exact trajectory taken by gradient descent: starting from $\boldsymbol{\theta} = [0, 0]$ (yellow dot) and converging to the optimum (white star). Notice how the steps are large at first (steep gradient) and shrink as the algorithm approaches the bowl's minimum.

#### Plot line of best fit

In [ ]:
# Plot dataset and GD line of best fit
pred_gd = np.dot(X, optimized_theta_vector)

plt.figure(figsize=(8, 5))
plt.scatter(X[:, 1], y, alpha=0.4, edgecolors='white', linewidths=0.5,
            color='steelblue', label='Training data', s=20)
plt.plot(X[:, 1], pred_gd, color='crimson', linewidth=2, label='GD solution')
plt.xlabel('Hours of study')
plt.ylabel('Test scores')
plt.title('Line of Best Fit — Gradient Descent')
plt.legend()
plt.tight_layout()
plt.show()

*   Does the initial values of theta affect your results? Why?
*   At what point does the lr become too large? How do you know it?
*   How does a very small lr affect to the learning process?

## Bonus 1: The Normal Equation (Closed-Form Solution)

Setting the gradient to zero gives $\mathbf{X}^\top\mathbf{X}\,\boldsymbol{\theta} = \mathbf{X}^\top\mathbf{y}$, so

$$\boldsymbol{\theta}^* = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$$

(derivation in section 7 of the theory notebook). In code: **1)** build $\mathbf{X}^\top\mathbf{X}$, **2)** build $\mathbf{X}^\top\mathbf{y}$, **3)** solve the linear system (`np.linalg.solve` is more stable than computing the inverse with `np.linalg.inv`).

In [ ]:
# Theta calculated from the Normal Equation: theta* = (X^T X)^{-1} X^T y
XtX = X.T @ X                                        # 1) (n+1) x (n+1) matrix
Xty = X.T @ y                                        # 2) (n+1) vector
theta_normal_equation = np.linalg.solve(XtX, Xty)    # 3) solve XtX theta = Xty

In [ ]:
ne_cost = compute_cost(theta_normal_equation, X, y)

print("-" * 45)
print("Normal Equation Results")
print("-" * 45)
print(f"  theta_0 (intercept) : {theta_normal_equation[0]:.6f}")
print(f"  theta_1 (slope)     : {theta_normal_equation[1]:.6f}")
print(f"  Minimised cost (MSE): {ne_cost:.6f}")
print("-" * 45)

In [ ]:
pred_ne = np.dot(X, theta_normal_equation)

plt.figure(figsize=(8, 5))
plt.scatter(X[:, 1], y, alpha=0.4, edgecolors='white', linewidths=0.5,
            color='steelblue', label='Training data', s=20)
plt.plot(X[:, 1], pred_ne, color='darkorange', linewidth=2, label='Normal Equation solution')
plt.xlabel('Hours of study')
plt.ylabel('Test scores')
plt.title('Line of Best Fit — Normal Equation')
plt.legend()
plt.tight_layout()
plt.show()

## Bonus 2: Vectorization
In order to make our code run faster, we can vectorize our implementation: replace the loop over examples by matrix operations.

$$\mathcal{J}(\boldsymbol{\theta}) = \frac{1}{2m}\|\mathbf{X}\boldsymbol{\theta} - \mathbf{y}\|^2$$

**1)** all predictions at once: $\mathbf{X}\boldsymbol{\theta}$, **2)** all errors at once: $\mathbf{X}\boldsymbol{\theta} - \mathbf{y}$, **3)** sum of squared errors: $\text{errors} \cdot \text{errors}$.

In [ ]:
def compute_cost_vectorized(theta, X, y):
    """
    vectorized version of compute_cost(theta, X, y)
    """
    ## number of training examples
    m = len(y)

    ## Calculate the cost with the given parameters
    predictions = X @ theta                  # 1) all predictions, shape (m,)
    errors = predictions - y                 # 2) all errors, shape (m,)
    J = (errors @ errors) / (2 * m)          # 3) sum of squared errors / 2m

    return J

In [ ]:
t0 = time.perf_counter()
J_loop = compute_cost(theta_normal_equation, X, y)
elapsed_loop = (time.perf_counter() - t0) * 1000

print(f"Loop version   — Cost: {J_loop:.6f}  |  Time: {elapsed_loop:.4f} ms")

In [ ]:
t0 = time.perf_counter()
J_vec = compute_cost_vectorized(theta_normal_equation, X, y)
elapsed_vec = (time.perf_counter() - t0) * 1000

print(f"Vectorized     — Cost: {J_vec:.6f}  |  Time: {elapsed_vec:.4f} ms")
print(f"Speed-up       : {elapsed_loop / elapsed_vec:.1f}×  (loop vs vectorized)")

We can also implement the same kind of vectorizaton to the gradient descent uptdates.
We start by measuring the time needed for the current implementation.

In [ ]:
t0 = time.perf_counter()
optimized_theta_vector, cost_graph, _ = gradient_descent(
    X, y, initial_theta_vector, learning_rate, num_iterations
)
elapsed_gd = (time.perf_counter() - t0) * 1000

print(f"  theta_0: {optimized_theta_vector[0]:.6f}")
print(f"  theta_1: {optimized_theta_vector[1]:.6f}")
print(f"  Time   : {elapsed_gd:.2f} ms")

And now rewrite the critical step of the algorithm: the gradient $\nabla_{\boldsymbol{\theta}}\mathcal{J} = \frac{1}{m}\mathbf{X}^\top(\mathbf{X}\boldsymbol{\theta} - \mathbf{y})$ in one line.

In [ ]:
def step_gradient(theta_current, X, y, alpha):
    """
    vectorized version of step_gradient(theta_current, X, y, alpha)
    """

    m = float(len(y))

    hypothesis_vector = X.dot(theta_current)

    loss = hypothesis_vector-y.squeeze()

    theta_gradient = X.T.dot(loss) / m

    theta_updated = theta_current - alpha*theta_gradient

    #Return updated parameters
    return theta_updated

In [ ]:
t0 = time.perf_counter()
optimized_theta_vector, cost_graph, _ = gradient_descent(
    X, y, initial_theta_vector, learning_rate, num_iterations
)
elapsed_vgd = (time.perf_counter() - t0) * 1000

print(f"  theta_0: {optimized_theta_vector[0]:.6f}")
print(f"  theta_1: {optimized_theta_vector[1]:.6f}")
print(f"  Time   : {elapsed_vgd:.2f} ms")
print(f"  Speed-up vs loop GD: {elapsed_gd / elapsed_vgd:.1f}×")

Please note that this is a very "small" model and dataset. Vectorizing becomes much more important (and helpful) for larger datasets and features spaces.

## Playing with the initial $\boldsymbol{\theta}$ and the learning rate

Let's answer the questions from above by experimenting. From now on we use the fast vectorized functions, so we can run many iterations.

In [ ]:
compute_cost = compute_cost_vectorized   # gradient_descent() will now use the fast cost

### 1. Different starting points → the same minimum (convexity)

We start gradient descent from 5 very different $\boldsymbol{\theta}$'s, using the same learning rate. If the cost is a **convex bowl** there is only one bottom, so every run must end at the same place: the Normal Equation solution.

In [ ]:
starts = [np.array([ 0.0,  0.0]),
          np.array([ 6.0,  5.0]),
          np.array([-5.0,  4.0]),
          np.array([ 7.0, -2.0]),
          np.array([-4.0, -2.5])]

# Cost surface on a grid that contains all the starting points
t0_range = np.linspace(-6, 8, 200)
t1_range = np.linspace(-3, 6, 200)
T0, T1 = np.meshgrid(t0_range, t1_range)
J_grid = np.array([compute_cost(np.array([a, b]), X, y)
                   for a, b in zip(T0.ravel(), T1.ravel())]).reshape(T0.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contour(T0, T1, J_grid, levels=np.logspace(0, 3, 25), cmap='viridis')

for k, theta_start in enumerate(starts):
    theta_end, _, path = gradient_descent(X, y, theta_start, learning_rate=0.03, num_iterations=1000)
    ax.plot(path[:, 0], path[:, 1], '.-', ms=3, lw=1, color=f'C{k}')
    ax.plot(*theta_start, 'o', color=f'C{k}', ms=9)
    print(f"start {theta_start}  ->  end {theta_end.round(3)}")

ax.plot(*theta_normal_equation, '*', color='crimson', ms=18, label='Normal Equation')
print(f"Normal Equation         ->      {theta_normal_equation.round(3)}")

ax.set_xlabel(r'$\theta_0$ (intercept)'); ax.set_ylabel(r'$\theta_1$ (slope)')
ax.set_title('Gradient descent from 5 starting points')
ax.legend(); plt.tight_layout(); plt.show()

All paths end at the red star. The starting point only changes **how long** it takes, not **where** we arrive: for a convex cost there are no local minima to get stuck in. (This is *not* true for neural networks, whose losses are not convex.)

Notice also the shape of the paths: they first move fast along $\theta_1$ (steep direction) and then crawl slowly along the long, flat valley in $\theta_0$. That is why 100 iterations with $\alpha = 0.01$ were not enough to get the intercept right earlier.

### 2. Choosing the learning rate

**Step 1: just try a few values** and look at the cost curve.

In [ ]:
learning_rates = [0.001, 0.01, 0.03, 0.058, 0.065]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for k, lr in enumerate(learning_rates):
    _, costs, path = gradient_descent(X, y, np.zeros(2), lr, num_iterations=100)
    axes[0].plot(costs, color=f'C{k}', label=fr'$\alpha$ = {lr}')
    if lr == 0.058:
        axes[1].plot(path[:, 0], path[:, 1], '.-', ms=3, lw=0.6, color=f'C{k}', label=fr'$\alpha$ = {lr}', zorder=2)
    if lr == 0.03:
        axes[1].plot(path[:, 0], path[:, 1], '.-', ms=5, lw=2, color=f'C{k}', label=fr'$\alpha$ = {lr}', zorder=3)

axes[0].set_yscale('log'); axes[0].set_ylim(1, 1e3); axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('MSE Cost (log scale)')
axes[0].set_title('Cost per iteration'); axes[0].legend()

axes[1].contour(T0, T1, J_grid, levels=np.logspace(0, 3, 25), cmap='viridis', alpha=0.6)
axes[1].set_xlim(-0.5, 3); axes[1].set_ylim(-0.5, 4.5)
axes[1].set_xlabel(r'$\theta_0$'); axes[1].set_ylabel(r'$\theta_1$')
axes[1].set_title('Small $\\alpha$ walks, large $\\alpha$ zig-zags'); axes[1].legend()
plt.tight_layout(); plt.show()

- $\alpha = 0.001$: the cost goes down, but very slowly (**too small**).
- $\alpha = 0.01,\ 0.03$: fast and smooth decrease (**good**).
- $\alpha = 0.058$: still converges, but it bounces from one side of the valley to the other (**on the edge**).
- $\alpha = 0.065$: every step overshoots more than the last one and the cost explodes, it leaves the plot after a few iterations (**too large, diverges**).

**Step 2: where exactly is the limit?** The theory (section 6 of the notes) says gradient descent converges iff

$$\alpha < \frac{2}{\lambda_{\max}}, \qquad \lambda_{\max} = \text{largest eigenvalue of } \tfrac{1}{m}\mathbf{X}^\top\mathbf{X}.$$

In [ ]:
H = X.T @ X / m                      # curvature of the bowl (the Hessian)
eigenvalues = np.linalg.eigvalsh(H)
alpha_max = 2 / eigenvalues.max()

print(f"Eigenvalues of H : {eigenvalues.round(3)}")
print(f"alpha must be < 2 / lambda_max = {alpha_max:.4f}")

**Step 3: a learning-rate sweep.** Run 100 iterations for many values of $\alpha$ and plot the final cost. This is also what you do in practice when you can't compute $\lambda_{\max}$ (e.g. neural networks): try values spaced ×3 apart (0.001, 0.003, 0.01, 0.03, …) and keep the **largest one whose cost still goes down smoothly**.

In [ ]:
alphas = np.logspace(-4, np.log10(0.07), 40)
final_costs = [gradient_descent(X, y, np.zeros(2), a, num_iterations=100)[1][-1] for a in alphas]

plt.figure(figsize=(8, 4.5))
plt.plot(alphas, final_costs, 'o-', ms=4)
plt.axvline(alpha_max, color='crimson', ls='--', label=fr'$2/\lambda_{{max}}$ = {alpha_max:.4f}')
plt.xscale('log'); plt.yscale('log'); plt.ylim(1.5, 100)
plt.text(alpha_max * 0.93, 50, 'diverges →', color='crimson', ha='right')
plt.xlabel(r'Learning rate $\alpha$'); plt.ylabel('Cost after 100 iterations')
plt.title('Learning-rate sweep')
plt.legend(); plt.tight_layout(); plt.show()

Reading the sweep from left to right: too small (high cost, not enough progress) → good (lowest cost) → past the red line the cost explodes. The best learning rate sits **just below** the limit, but in practice we stay a bit further away from it (e.g. half of it) to be safe.

**Summary**
- Initial $\boldsymbol{\theta}$: for linear regression it does not matter where you start. The cost is a convex bowl with a single minimum.
- $\alpha$ too small: slow. $\alpha$ too large: zig-zag, then divergence. The limit is $2/\lambda_{\max}$.
- How to tell: plot the cost per iteration. It should go down steadily; if it goes up or oscillates, lower $\alpha$.